# Twitter Sentiment vs Market Data Analysis

This notebook analyzes the relationship between daily Twitter sentiment and market indicators (IHSG and USD/IDR) for August-September 2025.

# Step 1: Setup and Load Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import matplotlib.dates as mdates

# Set professional styling
sns.set_style('whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['figure.titlesize'] = 18

# Consistent color scheme
colors = {'sentiment': '#1f77b4', 'ihsg': '#d62728', 'usd': '#2ca02c'}

# Load sentiment data
df_sent = pd.read_csv('data/daily_sentiment_gpt5.csv', parse_dates=['date'])
print('Sentiment data shape:', df_sent.shape)
print(df_sent.head())

# Load IHSG data
df_ihsg = pd.read_csv('data/ihsg_daily.csv', parse_dates=['Date'])
df_ihsg.rename(columns={'Date': 'date'}, inplace=True)
print('IHSG data shape:', df_ihsg.shape)
print(df_ihsg.head())

# Load USD/IDR data
df_usd = pd.read_csv('data/usd_idr_daily.csv', parse_dates=['Date'])
df_usd.rename(columns={'Date': 'date'}, inplace=True)
print('USD/IDR data shape:', df_usd.shape)
print(df_usd.head())

# Step 2: Preprocess Data

In [ ]:
# Standardize dates to YYYY-MM-DD
df_sent['date'] = pd.to_datetime(df_sent['date']).dt.strftime('%Y-%m-%d')
df_ihsg['date'] = pd.to_datetime(df_ihsg['date']).dt.strftime('%Y-%m-%d')
df_usd['date'] = pd.to_datetime(df_usd['date'], utc=True).dt.strftime('%Y-%m-%d')

# Filter to August-September 2025
start = '2025-08-01'
end = '2025-09-29'
df_sent = df_sent[(df_sent['date'] >= start) & (df_sent['date'] <= end)]
df_ihsg = df_ihsg[(df_ihsg['date'] >= start) & (df_ihsg['date'] <= end)]
df_usd = df_usd[(df_usd['date'] >= start) & (df_usd['date'] <= end)]

print('Filtered shapes:')
print('Sentiment:', df_sent.shape)
print('IHSG:', df_ihsg.shape)
print('USD/IDR:', df_usd.shape)

# Step 3: Compute Daily Returns

In [ ]:
# IHSG daily percentage returns
df_ihsg['ihsg_return'] = df_ihsg['Close'].pct_change() * 100

# USD/IDR daily percentage returns
df_usd['usd_return'] = df_usd['Close'].pct_change() * 100

print('IHSG returns sample:')
print(df_ihsg[['date', 'Close', 'ihsg_return']].head())
print('USD/IDR returns sample:')
print(df_usd[['date', 'Close', 'usd_return']].head())

# Step 4: Merge Datasets

In [ ]:
# Merge sentiment with IHSG
df_merged = pd.merge(df_sent, df_ihsg[['date', 'ihsg_return']], on='date', how='inner')

# Merge with USD/IDR
df_full = pd.merge(df_merged, df_usd[['date', 'usd_return']], on='date', how='inner')

# Convert date to datetime for plotting
df_full['date'] = pd.to_datetime(df_full['date'])

print('Merged data shape:', df_full.shape)
print(df_full.head())

# Step 5: Compute Correlations

In [ ]:
# Pearson correlations
corr_matrix = df_full[['net_sent', 'ihsg_return', 'usd_return']].corr()
print('Correlation Matrix:')
print(corr_matrix)

# Specific correlations
sent_ihsg_corr = corr_matrix.loc['net_sent', 'ihsg_return']
sent_usd_corr = corr_matrix.loc['net_sent', 'usd_return']
print(f'Net Sentiment vs IHSG Return: {sent_ihsg_corr:.3f}')
print(f'Net Sentiment vs USD/IDR Return: {sent_usd_corr:.3f}')

# Step 6: Analyze Lead-Lag Effects

In [ ]:
# Shift sentiment by 1-3 days
for lag in [1, 2, 3]:
    df_full[f'net_sent_lag{lag}'] = df_full['net_sent'].shift(lag)
    lag_corr_ihsg = df_full[[f'net_sent_lag{lag}', 'ihsg_return']].dropna().corr().iloc[0, 1]
    lag_corr_usd = df_full[[f'net_sent_lag{lag}', 'usd_return']].dropna().corr().iloc[0, 1]
    print(f'Lag {lag} days - Sentiment vs IHSG: {lag_corr_ihsg:.3f}, vs USD/IDR: {lag_corr_usd:.3f}')

print('Lead-lag analysis complete.')

# Step 7: Generate Presentable Plots

In [ ]:
# Run the presentable plots script
%run presentable_plots.py

# Interpretation for Report

- **Correlations**: Net sentiment shows moderate correlation with IHSG returns (e.g., 0.359). Check lead-lag for predictive power.
- **Lead-Lag**: If lagged correlations are higher, sentiment may lead market movements.
- **Limitations**: Small sample size, external factors (e.g., news) may confound results.
- **Implications**: Sentiment could inform trading strategies, but requires further validation.